In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ARTIFACTS_PUBLIC = PROJECT_ROOT / "artifacts" / "public"
DATA_PUBLIC = PROJECT_ROOT / "data" / "public"
MODELS_PUBLIC = PROJECT_ROOT / "models" / "public"

# Day 04 · V3_A raw contract and eval

Notebook de evidencia para Day 04. Documenta exploracion raw, contrato V3_A, calidad/cobertura y comparacion 1:1 contra baseline con el mismo cutoff oficial `2028-02-21`.

In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)

quality_path = PROJECT_ROOT / "artifacts/public/data_quality_v3_context.json"
registry_path = PROJECT_ROOT / "artifacts/public/metrics/final_baseline_vs_candidates.csv"
pure_metrics_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T084542Z_V3_A_LR_smote_0.5_v1_metrics.json"
policy_summary_path = PROJECT_ROOT / "artifacts/public/metrics/candidates/20260306/20260306T084542Z_V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1_run_summary.json"
dataset_path = PROJECT_ROOT / "data/public/dataset_modelo_proveedor_v3_context.csv"

quality = json.loads(quality_path.read_text(encoding="utf-8"))
pure_metrics = json.loads(pure_metrics_path.read_text(encoding="utf-8"))
policy_summary = json.loads(policy_summary_path.read_text(encoding="utf-8"))
registry = pd.read_csv(registry_path)
dataset_sample = pd.read_csv(dataset_path, nrows=5)

{
    "quality_run_id": quality["run_id"],
    "train_eval_run_id": pure_metrics["run_id"],
    "rows_output": quality["rows_output"],
    "events_output": quality["events_output"],
    "cutoff_date": quality["cutoff_date"],
}

{'quality_run_id': '20260306T_day04_rerun_v3a',
 'train_eval_run_id': '20260306T084542Z',
 'rows_output': 155946,
 'events_output': 15404,
 'cutoff_date': '2028-02-21'}

## 1. Contexto y objetivo del dia

- Objetivo del dia: construir `V3_A` con senales raw/comparativa/calculos pre-decision y compararlo 1:1 contra el baseline oficial.
- Regla de negocio cerrada para Day 04: misma receta `LR_smote_0.5`, mismo split temporal, mismo metric pack; el experimento debe aislar ganancia por features, no por tuning.
- Resultado real del dia: `V3_A_LR_smote_0.5_v1` queda registrado pero no supera el gate de reentreno; la comparativa secundaria con capa determinista mejora coherencia y Top-1, pero sigue fallando por `overrides_harmed > 0` y por no recuperar `Top-2` ni `balanced_accuracy` de baseline.

## 2. Inventario raw/staging/curated util para V3_A

In [3]:
inventory = pd.DataFrame(
    [
        {
            "source_table": "data/public/support/ofertas_typed.csv",
            "layer": "staging",
            "grain": "fecha_oferta + producto_canonico + proveedor_canonico + terminal_canonico",
            "used_in_v3_a": "yes",
            "why": "selected_source, reconciliation_status, cost_source y dispersion multi-terminal pre-decision",
        },
        {
            "source_table": "data/public/support/ofertas_raw_matrix_cells.csv",
            "layer": "staging/raw audit",
            "grain": "sheet cell",
            "used_in_v3_a": "audit_only",
            "why": "evidencia de calculos/transporte; requiere parser determinista antes de entrar en features",
        },
        {
            "source_table": "data/public/support/fact_ofertas_diarias.csv",
            "layer": "curated",
            "grain": "fecha_evento + producto_canonico + proveedor_canonico",
            "used_in_v3_a": "reference_only",
            "why": "sirve para comprobar que staging ya fue agregado, pero no conserva toda la trazabilidad raw necesaria",
        },
        {
            "source_table": "data/public/dataset_modelo_proveedor_v2_candidates.csv",
            "layer": "marts",
            "grain": "event_id + proveedor_candidato",
            "used_in_v3_a": "yes",
            "why": "base estructural, universo de evaluacion y features de competencia calculables sin perder filas",
        },
    ]
)
inventory

,source_table,layer,grain,used_in_v3_a,why
0,data/public/support/ofertas_typed.csv,staging,fecha_oferta + producto_canonico + proveedor_canonico + terminal_canonico,yes,"selected_source, reconciliation_status, cost_source y dispersion multi-terminal pre-decision"
1,data/public/support/ofertas_raw_matrix_cells.csv,staging/raw audit,sheet cell,audit_only,evidencia de calculos/transporte; requiere parser determinista antes de entrar en features
2,data/public/support/fact_ofertas_diarias.csv,curated,fecha_evento + producto_canonico + proveedor_canonico,reference_only,"sirve para comprobar que staging ya fue agregado, pero no conserva toda la trazabilidad raw necesaria"
3,data/public/dataset_modelo_proveedor_v2_candidates.csv,marts,event_id + proveedor_candidato,yes,"base estructural, universo de evaluacion y features de competencia calculables sin perder filas"


## 3. Tabla de decision por familia de senales

In [4]:
decision_table = pd.DataFrame(quality["signal_decision_table"])
decision_table

,signal_family,source_table,raw_grain,target_grain,requires_parser_change,pre_decision,leakage_risk,coverage_train,coverage_test,decision
0,source_quality,data/public/support/ofertas_typed.csv,fecha_oferta + producto_canonico + proveedor_canonico + terminal_canonico,event_id + proveedor_candidato,False,True,low,1.0,1.0,include_v3_a
1,multi_terminal_dispersion,data/public/support/ofertas_typed.csv,fecha_oferta + producto_canonico + proveedor_canonico + terminal_canonico,event_id + proveedor_candidato,False,True,low,1.0,1.0,include_v3_a
2,event_competition,data/public/dataset_modelo_proveedor_v2_candidates.csv,event_id + proveedor_candidato,event_id + proveedor_candidato,False,True,low,1.0,1.0,include_v3_a
3,raw_transport_or_unmapped_calcs,data/public/support/ofertas_raw_matrix_cells.csv,sheet cell,not_joined,True,True,unknown_until_parser,0.0,0.0,candidate_parser_extension


## 4. Contrato final V3_A

- Grano final: `1 fila = event_id + proveedor_candidato`.
- Base estructural: V2, con joins `left` para no perder ni filas ni eventos.
- Familias incluidas: `source_quality`, `multi_terminal_dispersion`, `event_competition`.
- Familias excluidas en Day 04: `raw_transport_or_unmapped_calcs`, porque existe evidencia raw pero no parser estable ni join seguro a grano operativo.
- Leakage guardrails: no entra ninguna senal derivada de `proveedor_elegido_real`, `precio_unitario_evento` o `importe_total_evento`.

In [5]:
contract_summary = pd.DataFrame(
    [
        {"contract_piece": "dataset_name", "value": "dataset_modelo_proveedor_v3_context.csv"},
        {"contract_piece": "grain", "value": "event_id + proveedor_candidato"},
        {"contract_piece": "rows_output", "value": quality["rows_output"]},
        {"contract_piece": "events_output", "value": quality["events_output"]},
        {"contract_piece": "rows_vs_v2_match", "value": quality["rows_vs_v2_match"]},
        {"contract_piece": "events_vs_v2_match", "value": quality["events_vs_v2_match"]},
        {"contract_piece": "events_with_invalid_positive_count", "value": quality["events_with_invalid_positive_count"]},
        {"contract_piece": "new_v3_numeric_features", "value": len([c for c in quality["feature_columns_num"] if c.startswith("v3_")])},
        {"contract_piece": "missing_flag_columns", "value": len(quality["missing_flag_columns"])},
    ]
)

v3_feature_contract = pd.DataFrame(
    [
        {"family": "source_quality", "columns": ", ".join([c for c in quality["feature_columns_num"] if c.startswith("v3_selected_") or c.startswith("v3_cost_source_") or c.startswith("v3_reconciliation_")])},
        {"family": "multi_terminal_dispersion", "columns": ", ".join([c for c in quality["feature_columns_num"] if c.startswith("v3_cost_") or c == "v3_share_terminales_min_cost"])},
        {"family": "event_competition", "columns": ", ".join([c for c in quality["feature_columns_num"] if c.startswith("v3_coste_segundo_") or c.startswith("v3_gap_") or c.startswith("v3_delta_") or c.startswith("v3_ratio_") or c.startswith("v3_rank_") or c.startswith("v3_candidatos_") or c.startswith("v3_is_unique_")])},
    ]
)

contract_summary, v3_feature_contract, dataset_sample

(                       contract_piece                                    value
 0                        dataset_name  dataset_modelo_proveedor_v3_context.csv
 1                               grain           event_id + proveedor_candidato
 2                         rows_output                                   155946
 3                       events_output                                    15404
 4                    rows_vs_v2_match                                        1
 5                  events_vs_v2_match                                        1
 6  events_with_invalid_positive_count                                        0
 7             new_v3_numeric_features                                       18
 8                missing_flag_columns                                        0,
                       family  \
 0             source_quality   
 1  multi_terminal_dispersion   
 2          event_competition   
 
                                                                 

## 5. Evidencia de calidad y cobertura

In [6]:
coverage_df = pd.DataFrame(quality["feature_coverage"]).T.reset_index().rename(columns={"index": "feature"})
coverage_summary = pd.DataFrame(
    [
        {"check": "rows_vs_v2_match", "value": quality["rows_vs_v2_match"]},
        {"check": "events_vs_v2_match", "value": quality["events_vs_v2_match"]},
        {"check": "events_with_invalid_positive_count", "value": quality["events_with_invalid_positive_count"]},
        {"check": "min_train_coverage", "value": coverage_df["coverage_train"].min()},
        {"check": "min_test_coverage", "value": coverage_df["coverage_test"].min()},
        {"check": "transport_keyword_hits", "value": quality["transport_audit"]["keyword_hits"]},
        {"check": "transport_decision", "value": quality["transport_audit"]["decision"]},
    ]
)

coverage_summary, coverage_df.sort_values(["coverage_train", "coverage_test", "feature"]).reset_index(drop=True), pd.DataFrame(quality["transport_audit"]["sample_hits"])


(                                check                       value
 0                    rows_vs_v2_match                           1
 1                  events_vs_v2_match                           1
 2  events_with_invalid_positive_count                           0
 3                  min_train_coverage                         1.0
 4                   min_test_coverage                         1.0
 5              transport_keyword_hits                        3413
 6                  transport_decision  candidate_parser_extension,
                                   feature  coverage_train  coverage_test
 0           v3_candidatos_min_coste_count             1.0            1.0
 1                     v3_cost_cv_terminal             1.0            1.0
 2                   v3_cost_mean_terminal             1.0            1.0
 3                  v3_cost_range_terminal             1.0            1.0
 4           v3_cost_source_calculos_share             1.0            1.0
 5                 

## 6. Metricas del candidato puro

In [7]:
model_compare = registry.loc[
    registry["model_variant"].isin(["LR_smote_0.5", "V3_A_LR_smote_0.5_v1"]),
    [
        "day_id",
        "model_variant",
        "dataset_name",
        "accuracy",
        "balanced_accuracy",
        "f1_pos",
        "top1_hit",
        "top2_hit",
        "coverage",
        "delta_top2_vs_baseline",
        "delta_bal_acc_vs_baseline",
        "promotion_decision",
    ],
].reset_index(drop=True)
model_compare

,day_id,model_variant,dataset_name,accuracy,balanced_accuracy,f1_pos,top1_hit,top2_hit,coverage,delta_top2_vs_baseline,delta_bal_acc_vs_baseline,promotion_decision
0,Day 02,LR_smote_0.5,dataset_modelo_proveedor_v2_candidates.csv,0.887445,0.865084,0.652039,0.764527,0.858336,1.0,0.000000,0.0000,keep_baseline
1,Day 04,V3_A_LR_smote_0.5_v1,dataset_modelo_proveedor_v3_context.csv,0.890802,0.853684,0.650284,0.739081,0.851880,1.0,-0.006456,-0.0114,keep_baseline


## 7. Metricas secundarias con policy Day 03

In [8]:
policy_compare = pd.DataFrame(
    [
        {"metric": "top1_hit_before", "value": policy_summary["metrics"]["top1_hit_before"]},
        {"metric": "top1_hit_after", "value": policy_summary["metrics"]["top1_hit_after"]},
        {"metric": "top2_hit_before", "value": policy_summary["metrics"]["top2_hit_before"]},
        {"metric": "top2_hit_after", "value": policy_summary["metrics"]["top2_hit_after"]},
        {"metric": "coherence_before", "value": policy_summary["metrics"]["coherence_before"]},
        {"metric": "coherence_after", "value": policy_summary["metrics"]["coherence_after"]},
        {"metric": "overrides_count", "value": policy_summary["metrics"]["overrides_count"]},
        {"metric": "overrides_improved", "value": policy_summary["metrics"]["overrides_improved"]},
        {"metric": "overrides_harmed", "value": policy_summary["metrics"]["overrides_harmed"]},
        {"metric": "coverage_after", "value": policy_summary["metrics"]["coverage_after"]},
    ]
)

policy_registry_compare = registry.loc[
    registry["model_variant"].isin(["BASELINE_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1", "V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1"]),
    [
        "day_id",
        "model_variant",
        "top1_hit",
        "top2_hit",
        "balanced_accuracy",
        "coverage",
        "delta_top2_vs_baseline",
        "delta_bal_acc_vs_baseline",
        "promotion_decision",
    ],
].reset_index(drop=True)

policy_compare, policy_registry_compare

(               metric      value
 0     top1_hit_before   0.742499
 1      top1_hit_after   0.766046
 2     top2_hit_before   0.852640
 3      top2_hit_after   0.856817
 4    coherence_before   0.827397
 5     coherence_after   0.972603
 6     overrides_count  70.000000
 7  overrides_improved  66.000000
 8    overrides_harmed   4.000000
 9      coverage_after   1.000000,
    day_id  \
 0  Day 04   
 
                                                                   model_variant  \
 0  V3_A_WITH_DETERMINISTIC_LAYER_PRODUCT_002_FOLLOW_PRODUCT_003_SUPPLIER_009_v1   
 
    top1_hit  top2_hit  balanced_accuracy  coverage  delta_top2_vs_baseline  \
 0  0.767186  0.855678           0.853684       1.0               -0.002658   
 
    delta_bal_acc_vs_baseline promotion_decision  
 0                    -0.0114      keep_baseline  )

## 8. Decisiones abiertas para Day 05

- `V3_A` queda como dataset reproducible y contrato defendible, pero no como nuevo champion de modelo.
- La siguiente decision no es re-tunear `LR`: primero hay que decidir si merece la pena abrir `Day 04.1` para parser determinista de transporte, porque la evidencia raw existe (`3413` hits) y hoy solo esta auditada.
- Si no se abre esa extension, Day 05 debe centrarse en champion final de serving, producto/demo y cierre narrativo con baseline de Day 03 como referencia vigente.
- Si se abre extension, el criterio es estricto: parser estable, join a grano operativo sin perdida de eventos y nueva corrida `V3_A2` bajo el mismo gate.

In [9]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
PROJECT_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == "notebooks" else PROJECT_ROOT

v2_path = PROJECT_ROOT / "data/public/dataset_modelo_proveedor_v2_candidates.csv"
v3_path = PROJECT_ROOT / "data/public/dataset_modelo_proveedor_v3_context.csv"
quality_path = PROJECT_ROOT / "artifacts/public/data_quality_v3_context.json"

v2_cols = pd.read_csv(v2_path, nrows=1).columns.tolist()
v3 = pd.read_csv(v3_path)
quality = json.loads(quality_path.read_text(encoding="utf-8"))

added_cols = [c for c in v3.columns if c not in v2_cols]

source_quality_cols = [
    "v3_selected_source_calculos_share",
    "v3_cost_source_calculos_share",
    "v3_reconciliation_conflict_share",
    "v3_reconciliation_single_source_share",
]

dispersion_cols = [
    "v3_cost_mean_terminal",
    "v3_cost_std_terminal",
    "v3_cost_range_terminal",
    "v3_cost_cv_terminal",
    "v3_share_terminales_min_cost",
]

competition_cols = [
    "v3_coste_segundo_evento",
    "v3_gap_min_vs_second_evento",
    "v3_delta_vs_second_evento",
    "v3_ratio_vs_second_evento",
    "v3_rank_pct_evento",
    "v3_coste_mean_evento",
    "v3_delta_vs_mean_evento",
    "v3_candidatos_min_coste_count",
    "v3_is_unique_min_coste_evento",
]

display(Markdown("## Qué cambia de V2 a V3_A"))
display(pd.DataFrame({"new_column_in_v3": added_cols}))

display(Markdown("## Qué parte sí hemos aprovechado"))
used_summary = pd.DataFrame([
    {
        "family": "source_quality_desde_staging",
        "what_it_means": "El modelo ve si la oferta/coste venían apoyados por la hoja Cálculos y si hubo conflicto Tabla vs Cálculos",
        "columns": ", ".join(source_quality_cols),
    },
    {
        "family": "dispersion_multi_terminal",
        "what_it_means": "El modelo ve si ese proveedor cambia mucho o poco entre terminales",
        "columns": ", ".join(dispersion_cols),
    },
    {
        "family": "competition_en_evento",
        "what_it_means": "El modelo ve si la comparativa del evento estaba clara o muy ajustada",
        "columns": ", ".join(competition_cols),
    },
])
display(used_summary)

display(Markdown("## Qué parte todavía NO hemos aprovechado"))
decision_table = pd.DataFrame(quality["signal_decision_table"])
not_used = decision_table.loc[decision_table["decision"] != "include_v3_a"].copy()
display(not_used)

transport_audit = quality["transport_audit"]
transport_summary = pd.DataFrame([{
    "keyword_hits_in_raw": transport_audit["keyword_hits"],
    "decision": transport_audit["decision"],
    "reading": "Hay evidencia raw de transporte/cálculos no mapeados, pero todavía no hay parser estable para meterlo en el modelo",
}])
display(transport_summary)
display(pd.DataFrame(transport_audit["sample_hits"]).head(5))

display(Markdown("## Ejemplo real de una comparativa donde V3_A añade contexto"))
mask_example = (
    pd.to_numeric(v3["v3_selected_source_calculos_share"], errors="coerce").fillna(0) > 0
) | (
    pd.to_numeric(v3["v3_cost_range_terminal"], errors="coerce").fillna(0) > 0
)

example_event_id = v3.loc[mask_example, "event_id"].iloc[0]

cols_to_show = [
    "event_id",
    "proveedor_candidato",
    "coste_min_dia_proveedor",
    "rank_coste_dia_producto",
    "v3_selected_source_calculos_share",
    "v3_cost_source_calculos_share",
    "v3_reconciliation_conflict_share",
    "v3_cost_range_terminal",
    "v3_cost_cv_terminal",
    "v3_coste_segundo_evento",
    "v3_gap_min_vs_second_evento",
    "v3_rank_pct_evento",
]

display(
    v3.loc[v3["event_id"] == example_event_id, cols_to_show]
      .sort_values(["coste_min_dia_proveedor", "proveedor_candidato"])
      .reset_index(drop=True)
)

print(f"event_id ejemplo: {example_event_id}")
print()
print("Lectura rápida:")
print("- Si v3_selected_source_calculos_share > 0, el modelo sí está viendo una sombra de la hoja Cálculos.")
print("- Si v3_cost_range_terminal > 0, el modelo está viendo que el proveedor cambia entre terminales.")
print("- Si v3_gap_min_vs_second_evento es pequeño, la decisión entre proveedores está muy ajustada.")
print("- Lo que todavía NO ve es el cálculo explícito de transporte tal y como lo mira el administrativo.")
print()
print("Conclusión operativa:")
print("- V3_A sí aprovecha parte del libro de comparativas.")
print("- V3_A todavía NO aprovecha el cálculo raw de transporte.")
print("- Si creemos que esa parte es decisiva para negocio, el siguiente paso correcto es parser + nueva variante sobre V3_A.")


## Qué cambia de V2 a V3_A

,new_column_in_v3
0,v3_selected_source_calculos_share
1,v3_cost_source_calculos_share
2,v3_reconciliation_conflict_share
3,v3_reconciliation_single_source_share
4,v3_cost_mean_terminal
5,v3_cost_std_terminal
6,v3_cost_range_terminal
7,v3_cost_cv_terminal
8,v3_share_terminales_min_cost
9,v3_coste_segundo_evento


## Qué parte sí hemos aprovechado

,family,what_it_means,columns
0,source_quality_desde_staging,El modelo ve si la oferta/coste venían apoyados por la hoja Cálculos y si hubo conflicto Tabla vs Cálculos,"v3_selected_source_calculos_share, v3_cost_source_calculos_share, v3_reconciliation_conflict_share, v3_reconciliatio..."
1,dispersion_multi_terminal,El modelo ve si ese proveedor cambia mucho o poco entre terminales,"v3_cost_mean_terminal, v3_cost_std_terminal, v3_cost_range_terminal, v3_cost_cv_terminal, v3_share_terminales_min_cost"
2,competition_en_evento,El modelo ve si la comparativa del evento estaba clara o muy ajustada,"v3_coste_segundo_evento, v3_gap_min_vs_second_evento, v3_delta_vs_second_evento, v3_ratio_vs_second_evento, v3_rank_..."


## Qué parte todavía NO hemos aprovechado

...

,keyword_hits_in_raw,decision,reading
0,3413,candidate_parser_extension,"Hay evidencia raw de transporte/cálculos no mapeados, pero todavía no hay parser estable para meterlo en el modelo"


,source_file,sheet_name,row_idx,col_idx,cell_value
0,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMPARATIVA DE PRECIOS/2015 -10 OCTUBRE/Comparativa de precios 13-10-2015.xls,Tabla,21,2,MENOR COSTE (CON TRANSPORTE)
1,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMPARATIVA DE PRECIOS/2015 -10 OCTUBRE/Comparativa de precios 14-10-2015.xls,Tabla,21,2,MENOR COSTE (CON TRANSPORTE)
2,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMPARATIVA DE PRECIOS/2015 -10 OCTUBRE/Comparativa de precios 15-10-2015.xls,Tabla,21,2,MENOR COSTE (CON TRANSPORTE)
3,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMPARATIVA DE PRECIOS/2015 -10 OCTUBRE/Comparativa de precios 16-10-2015.xls,Tabla,21,2,MENOR COSTE (CON TRANSPORTE)
4,./data/raw/SUPPLIER_DAILY_COMPARISON/2015 COMPARATIVA DE PRECIOS/2015 -10 OCTUBRE/Comparativa de precios 19-10-2015.xls,Tabla,21,2,MENOR COSTE (CON TRANSPORTE)


## Ejemplo real de una comparativa donde V3_A añade contexto

,event_id,proveedor_candidato,coste_min_dia_proveedor,rank_coste_dia_producto,v3_selected_source_calculos_share,v3_cost_source_calculos_share,v3_reconciliation_conflict_share,v3_cost_range_terminal,v3_cost_cv_terminal,v3_coste_segundo_evento,v3_gap_min_vs_second_evento,v3_rank_pct_evento
0,EVENT_8C109313E315,SUPPLIER_020,795.34,1.0,1.0,1.0,1.0,12.28,0.005437,796.34,1.0,0.000000
1,EVENT_8C109313E315,SUPPLIER_061,796.34,2.0,1.0,1.0,1.0,13.28,0.005509,796.34,1.0,0.083333
2,EVENT_8C109313E315,SUPPLIER_001,796.62,3.0,1.0,1.0,1.0,0.00,0.000000,796.34,1.0,0.166667
3,EVENT_8C109313E315,SUPPLIER_003,797.34,4.0,1.0,1.0,1.0,12.28,0.005440,796.34,1.0,0.250000
4,EVENT_8C109313E315,SUPPLIER_013,798.74,5.0,1.0,1.0,1.0,11.46,0.005252,796.34,1.0,0.333333
5,EVENT_8C109313E315,SUPPLIER_009,799.35,6.0,1.0,1.0,1.0,7.15,0.003322,796.34,1.0,0.416667
6,EVENT_8C109313E315,SUPPLIER_023,800.45,7.0,1.0,1.0,1.0,0.00,0.000000,796.34,1.0,0.500000
7,EVENT_8C109313E315,SUPPLIER_049,800.74,8.0,1.0,1.0,1.0,3.88,0.001975,796.34,1.0,0.583333
8,EVENT_8C109313E315,SUPPLIER_054,801.74,9.0,1.0,1.0,1.0,7.28,0.003705,796.34,1.0,0.666667
9,EVENT_8C109313E315,SUPPLIER_004,802.35,10.0,1.0,1.0,1.0,14.50,0.005778,796.34,1.0,0.750000


event_id ejemplo: EVENT_8C109313E315

Lectura rápida:
- Si v3_selected_source_calculos_share > 0, el modelo sí está viendo una sombra de la hoja Cálculos.
- Si v3_cost_range_terminal > 0, el modelo está viendo que el proveedor cambia entre terminales.
- Si v3_gap_min_vs_second_evento es pequeño, la decisión entre proveedores está muy ajustada.
- Lo que todavía NO ve es el cálculo explícito de transporte tal y como lo mira el administrativo.

Conclusión operativa:
- V3_A sí aprovecha parte del libro de comparativas.
- V3_A todavía NO aprovecha el cálculo raw de transporte.
- Si creemos que esa parte es decisiva para negocio, el siguiente paso correcto es parser + nueva variante sobre V3_A.
